# 04 — Phase 2 Group Experiments
Evaluasi covariate groups yang lolos Phase 1 screening menggunakan 4 model dengan hyperparameter Optuna. Evaluasi: simple 80/20 temporal split.

In [ ]:
import sys, unittest.mock; sys.modules.setdefault('transformers.dependency_versions_check', unittest.mock.MagicMock())

import pandas as pd
import numpy as np
import joblib
import gc
import traceback
import warnings
from datetime import datetime

from darts import TimeSeries
from darts.models import RandomForestModel, XGBModel, LightGBMModel
from darts.dataprocessing.transformers import Scaler, Diff
from darts.utils.missing_values import fill_missing_values
from sklearn.ensemble import ExtraTreesRegressor

try:
    from darts.models import SKLearnModel
except ImportError:
    from darts.models import RegressionModel as SKLearnModel

warnings.filterwarnings("ignore")

LEVEL_VARS = ["M2", "USDIDR", "Coal", "Copper", "Nickel", "Silver", "Tin", "STI", "Gold", "WTI", "GDP"]
RATE_VARS  = ["BI_Rate", "CPI", "NPL_Ratio", "US_Treasury_10Y"]
HORIZON    = 1
WINDOWS    = [30, 120]
TRAIN_RATIO = 0.8

# Load data
import glob
joblib_files = sorted(glob.glob("saved_models/df_merged_*.joblib"), reverse=True)
df_merged = joblib.load(joblib_files[0])
print(f"Loaded: {joblib_files[0]}")
print(f"Data: {df_merged.shape} | {df_merged['date'].min().date()} to {df_merged['date'].max().date()}")

# Load Optuna best params
TUNING = joblib.load("saved_models/optuna_tuning_results.joblib")
print("\nOptuna best params:")
for m, r in TUNING.items():
    print(f"  {m}: MAPE={r['best_value']:.2f}  params={r['best_params']}")


In [ ]:

# Load Phase 2 groups
GROUP_COVARIATES = joblib.load("saved_models/phase2_groups.joblib")
print("Phase 2 groups:")
for k, v in GROUP_COVARIATES.items():
    print(f"  {k}: {v}")

total_experiments = len(GROUP_COVARIATES) * 4 * len(WINDOWS)
print(f"\nTotal experiments: {total_experiments}")


## Helper: to_series, build_model, evaluate_model

In [ ]:

def to_series(df, target_col, covariates=None):
    target = TimeSeries.from_dataframe(
        df, time_col="date", value_cols=target_col,
        fill_missing_dates=True, freq="B",
    )
    target = fill_missing_values(target)
    cov = None
    if covariates:
        cov = TimeSeries.from_dataframe(
            df, time_col="date", value_cols=covariates,
            fill_missing_dates=True, freq="B",
        )
        cov = fill_missing_values(cov)
    return target, cov


def build_model(model_name, window, has_covariates):
    best_params = TUNING[model_name]['best_params'].copy()
    common = {
        "lags": window,
        "lags_past_covariates": window if has_covariates else None,
        "output_chunk_length": HORIZON,
    }
    if model_name == "RandomForest":
        return RandomForestModel(**common, random_state=42, n_jobs=-1, **best_params)
    elif model_name == "ExtraTrees":
        return SKLearnModel(
            **common,
            model=ExtraTreesRegressor(random_state=42, n_jobs=-1, **best_params),
        )
    elif model_name == "XGBoost":
        return XGBModel(**common, random_state=42, n_jobs=-1, **best_params)
    elif model_name == "LightGBM":
        return LightGBMModel(**common, random_state=42, n_jobs=-1, verbose=-1, **best_params)
    else:
        raise ValueError(f"Unknown model: {model_name}")


def transform_target(target_ts, split_idx):
    train_ts = target_ts[:split_idx]
    full_log  = target_ts.map(np.log)
    train_log = train_ts.map(np.log)
    diff = Diff(lags=1)
    train_log_diff = diff.fit_transform(train_log)
    full_log_diff  = diff.transform(full_log)
    scaler = Scaler()
    train_scaled = scaler.fit_transform(train_log_diff)
    full_scaled  = scaler.transform(full_log_diff)
    return train_scaled, full_scaled, scaler


def transform_covariates(cov_ts, split_idx):
    if cov_ts is None:
        return None, None
    train_cov = cov_ts[:split_idx]
    full_cov  = cov_ts
    cov_cols = cov_ts.components.tolist()
    level_cols = [c for c in cov_cols if c in LEVEL_VARS]
    rate_cols  = [c for c in cov_cols if c in RATE_VARS]
    parts_train, parts_full = [], []
    if level_cols:
        d = Diff(lags=1)
        parts_train.append(d.fit_transform(train_cov[level_cols].map(np.log)))
        parts_full.append(d.transform(full_cov[level_cols].map(np.log)))
    if rate_cols:
        d = Diff(lags=1)
        parts_train.append(d.fit_transform(train_cov[rate_cols]))
        parts_full.append(d.transform(full_cov[rate_cols]))
    ct = parts_train[0]
    cf = parts_full[0]
    for pt, pf in zip(parts_train[1:], parts_full[1:]):
        ct = ct.stack(pt)
        cf = cf.stack(pf)
    cov_scaler = Scaler()
    cov_scaler.fit(ct)
    full_cov_scaled = cov_scaler.transform(cf)
    return full_cov_scaled, ct.end_time()


def inverse_and_metrics(forecast_list, full_ts, scaler, split_idx):
    full_log = full_ts.map(np.log)
    all_dates, all_prices = [], []
    for chunk_scaled in forecast_list:
        chunk_diff = scaler.inverse_transform(chunk_scaled)
        dates = chunk_diff.time_index
        vals  = chunk_diff.values().flatten()
        idx = full_ts.get_index_at_point(dates[0])
        if idx == 0:
            continue
        anchor = full_log[idx - 1].values()[0][0]
        log_prices = anchor + np.cumsum(vals)
        all_dates.extend(dates)
        all_prices.extend(np.exp(log_prices))
    pred_df   = pd.DataFrame({"date": pd.to_datetime(all_dates), "predicted": all_prices})
    actual_df = full_ts.to_dataframe().reset_index()
    actual_df.columns = ["date", "actual"]
    eval_df = pd.merge(actual_df, pred_df, on="date", how="inner")
    y_true  = eval_df["actual"].values
    y_pred  = eval_df["predicted"].values
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    mae  = np.mean(np.abs(y_true - y_pred))
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - y_true.mean()) ** 2)
    r2   = 1 - (ss_res / ss_tot) if ss_tot > 0 else np.nan
    ad   = np.diff(y_true); pd_ = y_pred[1:] - y_true[:-1]
    da   = np.mean((ad > 0) == (pd_ > 0)) * 100 if len(ad) > 0 else np.nan
    return {"mape": round(mape,4), "mae": round(mae,4), "rmse": round(rmse,4),
            "r2": round(r2,4), "da": round(da,2)}


def evaluate_model(model_name, cov_name, cov_vars, window):
    target_ts, cov_ts = to_series(df_merged, "IHSG", cov_vars if cov_vars else None)
    n = len(target_ts)
    split_idx = int(n * TRAIN_RATIO)

    train_scaled, full_scaled, scaler = transform_target(target_ts, split_idx)
    full_cov_scaled, _ = transform_covariates(cov_ts, split_idx)

    model = build_model(model_name, window, has_covariates=bool(cov_vars))
    model.fit(train_scaled, past_covariates=full_cov_scaled)

    test_start = target_ts[split_idx].start_time()
    forecast_list = model.historical_forecasts(
        series=full_scaled,
        past_covariates=full_cov_scaled,
        start=test_start,
        forecast_horizon=HORIZON,
        stride=HORIZON,
        retrain=False,
        last_points_only=False,
        verbose=False,
    )
    if isinstance(forecast_list, TimeSeries):
        forecast_list = [forecast_list]
    return inverse_and_metrics(forecast_list, target_ts, scaler, split_idx)


## Phase 2: Experiment Loop

In [ ]:

MODEL_NAMES = ["RandomForest", "ExtraTrees", "XGBoost", "LightGBM"]
results = []
failed = []

total = len(MODEL_NAMES) * len(GROUP_COVARIATES) * len(WINDOWS)
done  = 0

for model_name in MODEL_NAMES:
    for cov_name, cov_vars in GROUP_COVARIATES.items():
        for window in WINDOWS:
            done += 1
            tag = f"{model_name} | {cov_name} | W{window}_H1"
            print(f"[{done}/{total}] {tag}", end=" ... ", flush=True)
            try:
                m = evaluate_model(model_name, cov_name, cov_vars, window)
                results.append({
                    "Model": model_name, "Covariates": cov_name, "Window": window,
                    **m
                })
                print(f"MAPE={m['mape']:.4f}%")
            except Exception as e:
                failed.append({"tag": tag, "error": str(e)})
                print(f"FAILED: {e}")
                traceback.print_exc()
            finally:
                gc.collect()

print(f"\nDone: {len(results)} OK, {len(failed)} failed")
if failed:
    print("Failed:", [f['tag'] for f in failed])


In [ ]:

df_p2 = pd.DataFrame(results)
df_p2.to_csv("phase2_group_results.csv", index=False)
print("Saved: phase2_group_results.csv")
print(df_p2.sort_values("MAPE").to_string(index=False))


In [ ]:

# Find best overall configuration
best = df_p2.sort_values("MAPE").iloc[0]
print("\nBest Phase 2 configuration:")
print(f"  Model      : {best['Model']}")
print(f"  Covariates : {best['Covariates']}")
print(f"  Window     : {best['Window']}")
print(f"  MAPE       : {best['MAPE']:.4f}%")
print(f"  RMSE       : {best['RMSE']:.2f}")
print(f"  DA         : {best['DA']:.1f}%")

# Save best config for SHAP notebook
best_config = best.to_dict()
joblib.dump(best_config, "saved_models/best_config_phase2.joblib")
print("\nSaved best config: saved_models/best_config_phase2.joblib")
